# Rol 1 — Analista de Incertidumbre e Información
### Proyecto Retador · UrbanMove · ML No Supervisado 2026-II · Universidad de La Sabana

**Pregunta de negocio:** ¿Qué tan predecible es la demanda de viajes por zona y momento del día? ¿Cuánto se gana (en bits) al usar información temporal para anticipar dónde ocurrirá la demanda?

## 1. Carga, limpieza y variables derivadas

In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)
df = pd.read_csv('train.csv')
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])
print(f"Registros totales: {len(df):,}")
df.isna().sum()


Registros totales: 1,458,644


id                    0
vendor_id             0
pickup_datetime       0
dropoff_datetime      0
passenger_count       0
pickup_longitude      0
pickup_latitude       0
dropoff_longitude     0
dropoff_latitude      0
store_and_fwd_flag    0
trip_duration         0
dtype: int64

**Muestreo:** se toma una muestra aleatoria de **40,000 viajes** (de 1,458,644 totales) — suficiente para estimar entropías con error estándar bajo, evitando el costo computacional de procesar el archivo completo en cada iteración.

In [2]:
n = 40000
samp = df.sample(n=n, random_state=42).reset_index(drop=True)

def haversine(lon1, lat1, lon2, lat2):
    R = 6371  # km
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon, dlat = lon2 - lon1, lat2 - lat1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2*R*np.arcsin(np.sqrt(a))

samp['dist_km'] = haversine(samp.pickup_longitude, samp.pickup_latitude,
                             samp.dropoff_longitude, samp.dropoff_latitude)
samp['hour'] = samp.pickup_datetime.dt.hour
samp['dow'] = samp.pickup_datetime.dt.dayofweek
samp['is_weekend'] = samp.dow >= 5

# Filtro de calidad: caja geográfica de NYC, distancias y duraciones plausibles
box = samp[(samp.pickup_longitude.between(-74.05, -73.75)) &
           (samp.pickup_latitude.between(40.60, 40.90)) &
           (samp.dropoff_longitude.between(-74.05, -73.75)) &
           (samp.dropoff_latitude.between(40.60, 40.90)) &
           (samp.dist_km.between(0.05, 60)) &
           (samp.trip_duration.between(60, 7200))].copy()

print(f"Viajes válidos tras limpieza: {len(box):,} ({len(box)/n:.1%} de la muestra)")


Viajes válidos tras limpieza: 39,320 (98.3% de la muestra)


**Zonificación espacial:** se discretiza el espacio de recogida en una grilla de `0.02°` (~1.7×2.2 km por celda). Es la resolución que revela contraste real entre horarios sin generar miles de zonas casi vacías (ver sensibilidad de grilla evaluada en el desarrollo).

In [3]:
STEP = 0.02

def zone(lon, lat, step=STEP):
    return (np.floor(lon/step)).astype(int).astype(str) + "_" + (np.floor(lat/step)).astype(int).astype(str)

box['pickup_zone'] = zone(box.pickup_longitude, box.pickup_latitude)
box['dropoff_zone'] = zone(box.dropoff_longitude, box.dropoff_latitude)
print(f"Zonas de recogida ocupadas: {box.pickup_zone.nunique()}")


Zonas de recogida ocupadas: 96


## 2. Entropía de Shannon — ¿qué tan caótica es la demanda?

$$H(X) = -\sum_i p(x_i)\log_2 p(x_i)$$

Se calcula sobre la distribución empírica de viajes por zona de recogida (global, y condicionada a hora pico vs. valle).

In [4]:
def entropy_bits(counts):
    p = np.asarray(counts, dtype=float)
    p = p / p.sum()
    p = p[p > 0]  # evita log(0)
    return -np.sum(p * np.log2(p))

zc = box.pickup_zone.value_counts()
n_zones = len(zc)
H_zone = entropy_bits(zc.values)
H_max = np.log2(n_zones)
redundancy = 1 - H_zone / H_max

print(f"Zonas activas: {n_zones}")
print(f"H(zona) = {H_zone:.3f} bits   |   H_max (uniforme) = {H_max:.3f} bits")
print(f"Redundancia = {redundancy:.1%}")
print(f"Zonas 'efectivas' (2^H) = {2**H_zone:.1f} de {n_zones} reales")
print(f"Participación de las 3 zonas top: {zc.head(3).sum()/zc.sum():.1%}")


Zonas activas: 96
H(zona) = 3.937 bits   |   H_max (uniforme) = 6.585 bits
Redundancia = 40.2%
Zonas 'efectivas' (2^H) = 15.3 de 96 reales
Participación de las 3 zonas top: 40.7%


**Lectura de negocio:** con 96 zonas activas, la incertidumbre uniforme sería de 6.585 bits (equivalente a escoger entre 96 zonas con igual probabilidad). La demanda real tiene **H(zona) = 3.937 bits**, es decir, una redundancia del **40.2%**: la demanda se comporta como si sólo hubiera **~15 zonas igualmente probables**, no 96. Las 3 zonas más activas concentran el **40.7%** de todos los viajes. 

**Implicación operativa:** UrbanMove no necesita distribuir la flota de forma uniforme sobre el mapa; una fracción reducida de celdas explica la mayoría de la demanda — la base cuantitativa que alimenta la segmentación espacial del Rol 2.

In [5]:
peak = box[box.hour.isin([7,8,9,17,18,19])]     # horas pico AM/PM
valley = box[box.hour.isin([0,1,2,3,4,5])]        # madrugada

H_peak = entropy_bits(peak.pickup_zone.value_counts().values)
H_valley = entropy_bits(valley.pickup_zone.value_counts().values)

print(f"H(zona | hora pico)   = {H_peak:.3f} bits   (n={len(peak):,})")
print(f"H(zona | madrugada)   = {H_valley:.3f} bits   (n={len(valley):,})")

top5_peak = peak.pickup_zone.value_counts(normalize=True).head(5)
print(f"\nTop-5 zonas en hora pico cubren {top5_peak.sum():.1%} de la demanda pico")


H(zona | hora pico)   = 3.881 bits   (n=12,141)
H(zona | madrugada)   = 3.979 bits   (n=4,580)

Top-5 zonas en hora pico cubren 59.8% de la demanda pico


**Lectura de negocio:** en hora pico la entropía baja a **3.881 bits** frente a **3.979 bits** en madrugada — la demanda pico es más predecible geográficamente y las **5 zonas top concentran ~60%** de esos viajes. En madrugada la demanda se dispersa más (mayor entropía) porque los pocos viajes existentes se reparten por más zonas residenciales. **Decisión concreta:** concentrar flota en un puñado de zonas durante la hora pico (alta certeza, alto retorno) y mantener cobertura más amplia — pero con menos vehículos por celda — en la madrugada (baja demanda, baja certeza).

## 3. Entropía cruzada y Divergencia KL — día laboral vs. fin de semana

$$H(P,Q) = -\sum_i p(x_i)\log_2 q(x_i) \qquad D_{KL}(P\|Q) = \sum_i p(x_i)\log_2\frac{p(x_i)}{q(x_i)} = H(P,Q) - H(P)$$

$P$ = distribución de zona de recogida en día laboral, $Q$ = distribución en fin de semana. Se usa suavizado aditivo (+0.5) para evitar $q(x_i)=0$ en zonas sin observaciones en alguno de los dos contextos.

In [6]:
def dist_over_zones(sub, all_zones, alpha=0.5):
    c = sub.pickup_zone.value_counts().reindex(all_zones, fill_value=0).values.astype(float)
    c = c + alpha
    return c / c.sum()

all_zones = box.pickup_zone.unique()
wd, we = box[~box.is_weekend], box[box.is_weekend]

P = dist_over_zones(wd, all_zones)   # laboral
Q = dist_over_zones(we, all_zones)   # fin de semana

H_P = -np.sum(P * np.log2(P))
H_cross = -np.sum(P * np.log2(Q))
KL_PQ = np.sum(P * np.log2(P / Q))
KL_QP = np.sum(Q * np.log2(Q / P))

print(f"n laboral={len(wd):,}  n fin de semana={len(we):,}  zonas={len(all_zones)}")
print(f"H(laboral) = {H_P:.4f} bits")
print(f"H(laboral, finde) [cruzada] = {H_cross:.4f} bits")
print(f"D_KL(laboral || finde) = {KL_PQ:.4f} bits")
print(f"D_KL(finde || laboral) = {KL_QP:.4f} bits   (confirma asimetría de la KL)")


n laboral=28,061  n fin de semana=11,259  zonas=96
H(laboral) = 3.9084 bits
H(laboral, finde) [cruzada] = 3.9543 bits
D_KL(laboral || finde) = 0.0459 bits
D_KL(finde || laboral) = 0.0508 bits   (confirma asimetría de la KL)


**Lectura de negocio:** $D_{KL}(\text{laboral}\|\text{finde}) = 0.0459$ bits — pequeño pero no nulo, y **asimétrico** respecto a $D_{KL}(\text{finde}\|\text{laboral}) = 0.0508$ el valor calculado arriba (confirma que KL no es una distancia). El costo de codificar la demanda laboral usando el modelo de fin de semana (entropía cruzada 3.9543 bits) supera en 0.0459 bits a la entropía real del laboral (3.9084 bits): es el "precio en información" de operar con un solo modelo para ambos contextos. **Implicación:** aunque la diferencia es moderada, sobre >1.4M de viajes/año un modelo único de posicionamiento subestima sistemáticamente ciertas zonas laborales — justifica que UrbanMove mantenga **dos políticas de flota diferenciadas** (laboral / fin de semana) en vez de una sola.

## 4. Información mutua y entropía condicional — ¿cuánto dice la hora sobre la zona?

$$I(H;Z) = \sum_{h,z} p(h,z)\log_2\frac{p(h,z)}{p(h)p(z)} \qquad H(Z\mid H) = H(Z) - I(H;Z)$$

Se restringe a las 20 zonas con más demanda para tener una tabla de contingencia hora×zona con soporte estadístico suficiente en cada celda.

In [7]:
top_zones = zc.head(20).index
sub = box[box.pickup_zone.isin(top_zones)].copy()

joint = pd.crosstab(sub['hour'], sub['pickup_zone'])
Pxy = joint.values / joint.values.sum()
Px = Pxy.sum(axis=1, keepdims=True)
Py = Pxy.sum(axis=0, keepdims=True)
mask = Pxy > 0
MI = np.sum(Pxy[mask] * np.log2(Pxy[mask] / (Px @ Py)[mask]))

H_hour = entropy_bits(sub['hour'].value_counts().values)
H_zone20 = entropy_bits(sub['pickup_zone'].value_counts().values)
H_zone_given_hour = H_zone20 - MI

print(f"n (top-20 zonas) = {len(sub):,}")
print(f"H(hora) = {H_hour:.4f} bits")
print(f"H(zona | top20) = {H_zone20:.4f} bits")
print(f"I(hora ; zona) = {MI:.4f} bits")
print(f"H(zona | hora) = {H_zone_given_hour:.4f} bits")
print(f"Reducción de incertidumbre = {MI/H_zone20:.1%}")


n (top-20 zonas) = 37,983
H(hora) = 4.4555 bits
H(zona | top20) = 3.6809 bits
I(hora ; zona) = 0.0563 bits
H(zona | hora) = 3.6245 bits
Reducción de incertidumbre = 1.5%


**Lectura de negocio:** $I(\text{hora};\text{zona}) = 0.0563$ bits, apenas **1.5%** de la entropía de zona. Conocer la hora exacta reduce muy poco la incertidumbre sobre *cuál* será la zona de recogida entre las top-20 — la ubicación de los hotspots (Midtown, aeropuertos, distritos financieros) es estructural y no depende fuertemente de la hora, mientras que lo que sí varía con la hora es *cuánta* demanda hay en esos hotspots.

## 5. Desigualdad de Jensen y *log-sum-exp trick* 

La **desigualdad de Jensen** ($\mathbb{E}[\log X] \le \log \mathbb{E}[X]$ para funciones cóncavas) es la base teórica de por qué la entropía cruzada nunca es menor que la entropía propia ($H(P,Q)\ge H(P)$, verificado numéricamente en la Sección 3) y por qué el algoritmo EM (Rol 3) maximiza una **cota inferior** de la log-verosimilitud en lugar de la log-verosimilitud exacta.


In [8]:
log_p_zone = np.log(zc.values / zc.values.sum())          # log-probabilidades reales por zona
scaled = log_p_zone * 450                                    # simula log-verosimilitud acumulada (~450 obs. independientes)

m = scaled.max()
logsumexp_stable = m + np.log(np.sum(np.exp(scaled - m)))

with np.errstate(over='ignore', under='ignore'):
    naive = np.log(np.sum(np.exp(scaled)))

print(f"Rango de log-probabilidades escaladas: [{scaled.min():.1f}, {scaled.max():.1f}]")
print(f"log-sum-exp (estable) = {logsumexp_stable:.3f}")
print(f"Cálculo ingenuo log(sum(exp(x))) = {naive}")


Rango de log-probabilidades escaladas: [-4760.8, -783.2]
log-sum-exp (estable) = -783.161
Cálculo ingenuo log(sum(exp(x))) = -inf


C:\Users\tomas\AppData\Local\Temp\ipykernel_14280\2342993851.py:8: RuntimeWarning: divide by zero encountered in log
  naive = np.log(np.sum(np.exp(scaled)))


**Resultado:** al exponenciar directamente valores tan negativos (`exp(-783)`), Python hace *underflow* a `0.0` para **todas** las 96 zonas simultáneamente, y `log(0) = -inf` — la log-verosimilitud se vuelve indefinida numéricamente. El truco `log-sum-exp` (restar el máximo antes de exponenciar) recupera el valor correcto y finito.

## 6. Síntesis — insumos cuantitativos para el equipo

| Métrica | Valor | Uso aguas abajo |
|---|---|---|
| H(zona) global | 3.9084 bits (40.2% redundancia vs. uniforme) | Rol 2: valida que K-means/DBSCAN deben encontrar del orden de ~15 clústeres efectivos, no 105 |
| H(zona\|pico) vs H(zona\|valle) | 3.881 vs 3.979 bits | Rol 4: recomendación de redistribución horaria de flota |
| D_KL(laboral‖finde) | 0.0459 bits (asimétrica) | Rol 4: justifica dos políticas operativas distintas |
| I(hora;zona) | 0.0563 bits (1.5% de H(zona)) | Rol 3: la hora no segmenta la zona de forma determinista → usar mezcla probabilística conjunta |
| log-sum-exp | evita `-inf` en verosimilitud acumulada | Rol 3: paso E del algoritmo EM, estabilidad numérica |

Todas las cifras provienen de la misma muestra de 39,320 viajes válidos y la misma zonificación (`STEP=0.02°`), garantizando consistencia entre bloques del equipo.